In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
DATA_DIR = Path("data")

FILE_STATES = DATA_DIR / "states_ilinet_2010_2017.csv"
FILE_REGIONS = DATA_DIR / "hhs_ilinet_2002_2017.csv"

In [26]:
def show_df(df, show_full=False):
    if show_full:
        pd.set_option("display.max_rows", None)
        pd.set_option("display.max_columns", None)
        display(df)
        pd.reset_option("display.max_rows")
        pd.reset_option("display.max_columns")
    else:
        display(df)

In [36]:
df_states = pd.read_csv(FILE_STATES)
df_regions = pd.read_csv(FILE_REGIONS)
df_states = df_states[df_states['REGION']!="dc"]

In [41]:
def generate_input_npy(df, output_path):
    df = df.copy()
    df["TIME"] = (
        df["YEAR"].astype(int).astype(str)
        + "-"
        + df["WEEK"].astype(int).astype(str).str.zfill(2)
    )

    times   = sorted(df["TIME"].unique())
    regions = sorted(df["REGION"].unique())

    T = len(times)
    N = len(regions)
    feature_cols = ["ILITOTAL"]
    F = len(feature_cols)

    data = np.zeros((T, N, F), dtype=np.float32)

    time_index   = {t: i for i, t in enumerate(times)}
    region_index = {r: i for i, r in enumerate(regions)}

    for _, row in df.iterrows():
        ti = time_index[row["TIME"]]
        ri = region_index[row["REGION"]]
        for fi, col in enumerate(feature_cols):
            val = row[col]
            data[ti, ri, fi] = 0.0 if pd.isna(val) else float(val)

    np.save(output_path, data)
    return data

states_arr = generate_input_npy(df_states, DATA_DIR/"ilinet_states")
regions_arr = generate_input_npy(df_regions, DATA_DIR/"ilinet_regions")